# Snowflake Trust Center → Google SecOps

查詢 Snowflake Trust Center findings，產生格式化 Excel 報表，並透過 Google SecOps webhook feed (`:importPushLogs`) 送出。

**支援功能**
- 多帳號 (multi-account) consolidated 報表
- AT_RISK_ENTITIES 詳細展開
- 4 張 matplotlib 圖表（severity 分布、top scanners、帳號 heatmap、時間趨勢）
- 多 sheet 格式化 Excel export（含嵌入圖表）
- Google SecOps UDM JSON lines webhook 送出，含 retry 機制與 4 MB request cap

## 前置需求

| 項目 | 說明 |
|------|------|
| Snowflake edition | Enterprise 或以上（Trust Center 需要） |
| Snowflake role | `TRUST_CENTER_VIEWER` database role 或 `ACCOUNTADMIN` |
| Snowflake key pair | 可供 `SNOWFLAKE_JWT` 使用的 private key，且 public key 已綁到對應 user |
| Google SecOps | Chronicle webhook feed endpoint (`:importPushLogs`)，可用 URL query 或 header auth |
| Python | 3.10+ |

## 執行流程

```
Cell 2  → 安裝 dependencies
Cell 4  → 設定 CONFIG（填 secrets）
Cell 6  → 連接 Snowflake（自動驗證 Trust Center schema 存在）
Cell 10 → 抓取所有帳號的 Trust Center findings
Cell 12 → 展開 AT_RISK_ENTITIES detail
Cell 14 → 轉換成 Google SecOps UDM JSON lines event
Cell 16 → 產生 matplotlib 圖表（4 張）
Cell 18 → Export 格式化 Excel 報表（含圖表）
Cell 20 → 送出 webhook 到 Google SecOps（支援 DRY_RUN）
Cell 22 → Verification summary
Cell 24 → 關閉 Snowflake connections
```


In [ ]:
# Cell 2 - Install dependencies
%pip install --quiet "snowflake-connector-python[pandas]>=3.16.0" "requests>=2.31.0" "python-dotenv>=1.0.0" "tenacity>=8.2.0" "pandas>=2.0.0" "openpyxl>=3.1.2" "matplotlib>=3.8.0" "ipywidgets>=8.0.0"
print("Dependencies installed.")


## Step 1 — Configuration

設定方式（優先順序）：
1. 在 `SNOW/` 目錄建立 `.env` 檔（參考 `.env.example`）
2. 設定環境變數
3. 執行此 cell 時會互動式詢問（private key passphrase、webhook secrets 用 `getpass`）

多帳號：在 `.env` 中設定 `SNOWFLAKE_ACCOUNT_1`, `SNOWFLAKE_ACCOUNT_2` … 留空或不設定則跳過。

Google SecOps auth：
- 若 `SECOPS_WEBHOOK_URL` 已含 `?key=...&secret=...`，sender 直接使用 URL。
- 若 URL 未含認證參數，則改用 `SECOPS_API_KEY` + `SECOPS_WEBHOOK_SECRET` headers。


In [ ]:
# Cell 4 - Configuration
import getpass
import os
from pathlib import Path
from urllib.parse import parse_qs, urlparse

from dotenv import load_dotenv

NOTEBOOK_FILENAME = "snowflake_trust_center_to_secops.ipynb"


def _detect_project_dir() -> Path:
    """Resolve the notebook directory or fall back to the current working directory."""
    override = os.getenv("NOTEBOOK_ROOT")
    if override:
        return Path(override).expanduser().resolve()

    search_roots = [Path.cwd(), *Path.cwd().parents]
    for root in search_roots:
        if (root / NOTEBOOK_FILENAME).exists():
            return root.resolve()
    return Path.cwd().resolve()


def _resolve_private_key_path(raw_path: str) -> str:
    """Normalize the configured private key path for display and connector use."""
    raw_path = (raw_path or "").strip()
    if not raw_path:
        return ""
    return str(Path(raw_path).expanduser().resolve())


def _url_has_query_credential(url: str, key: str) -> bool:
    """Return True when the webhook URL already carries the requested auth parameter."""
    values = parse_qs(urlparse(url).query).get(key, [])
    return any(value.strip() for value in values)


PROJECT_DIR = _detect_project_dir()
ENV_PATH = Path(os.getenv("CONFIG_ENV_PATH", PROJECT_DIR / ".env")).expanduser().resolve()
load_dotenv(ENV_PATH)


def _load_accounts() -> list[dict]:
    """Load account settings from environment variables and interactive prompts."""
    accounts = []
    for i in range(1, 10):
        acct = os.getenv(f"SNOWFLAKE_ACCOUNT_{i}", "").strip()
        if not acct:
            if i == 1:
                acct = input("Snowflake account locator (e.g. xy12345.us-east-1): ").strip()
                if not acct:
                    raise ValueError("At least one Snowflake account is required.")
            else:
                break

        user = os.getenv(f"SNOWFLAKE_USER_{i}", "").strip()
        if i == 1 and not user:
            user = input("Snowflake user: ").strip()

        private_key_path = _resolve_private_key_path(
            os.getenv(f"SNOWFLAKE_PRIVATE_KEY_PATH_{i}", "")
        )
        if i == 1 and not private_key_path:
            private_key_path = _resolve_private_key_path(
                input("Snowflake private key path: ").strip()
            )

        private_key_passphrase = os.getenv(
            f"SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_{i}", ""
        )
        if i == 1 and not os.getenv(f"SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_{i}"):
            private_key_passphrase = getpass.getpass(
                "Snowflake private key passphrase (leave blank if not encrypted): "
            )

        accounts.append(
            {
                "account": acct,
                "user": user,
                "private_key_path": private_key_path,
                "private_key_passphrase": private_key_passphrase,
                "role": os.getenv(f"SNOWFLAKE_ROLE_{i}", "TRUST_CENTER_VIEWER").strip()
                or "TRUST_CENTER_VIEWER",
                "warehouse": os.getenv(f"SNOWFLAKE_WAREHOUSE_{i}", "COMPUTE_WH").strip()
                or "COMPUTE_WH",
                "label": os.getenv(f"SNOWFLAKE_LABEL_{i}", f"account-{i}").strip()
                or f"account-{i}",
            }
        )
    return accounts


CONFIG = {
    "SNOWFLAKE_ACCOUNTS": _load_accounts(),
    "LOOKBACK_DAYS": int(os.getenv("LOOKBACK_DAYS", "7")),
    "SECOPS_WEBHOOK_URL": os.getenv("SECOPS_WEBHOOK_URL", "").strip()
    or input("Google SecOps webhook URL: ").strip(),
    "SECOPS_API_KEY": os.getenv("SECOPS_API_KEY", "").strip(),
    "SECOPS_WEBHOOK_SECRET": os.getenv("SECOPS_WEBHOOK_SECRET", "").strip(),
    "BATCH_SIZE": int(os.getenv("BATCH_SIZE", "100")),
    "DRY_RUN": os.getenv("DRY_RUN", "false").lower() == "true",
    "EXPORT_PATH": os.getenv("EXPORT_PATH", "./trust_center_report.xlsx"),
}

if not CONFIG["DRY_RUN"] and not _url_has_query_credential(CONFIG["SECOPS_WEBHOOK_URL"], "key"):
    CONFIG["SECOPS_API_KEY"] = CONFIG["SECOPS_API_KEY"] or getpass.getpass(
        "Google SecOps API key: "
    )
if not CONFIG["DRY_RUN"] and not _url_has_query_credential(
    CONFIG["SECOPS_WEBHOOK_URL"], "secret"
):
    CONFIG["SECOPS_WEBHOOK_SECRET"] = (
        CONFIG["SECOPS_WEBHOOK_SECRET"]
        or getpass.getpass("Google SecOps webhook secret: ")
    )

webhook_has_key = _url_has_query_credential(CONFIG["SECOPS_WEBHOOK_URL"], "key")
webhook_has_secret = _url_has_query_credential(CONFIG["SECOPS_WEBHOOK_URL"], "secret")
missing_secops_auth = []
if not CONFIG["DRY_RUN"] and not webhook_has_key and not CONFIG["SECOPS_API_KEY"]:
    missing_secops_auth.append("SECOPS_API_KEY")
if not CONFIG["DRY_RUN"] and not webhook_has_secret and not CONFIG["SECOPS_WEBHOOK_SECRET"]:
    missing_secops_auth.append("SECOPS_WEBHOOK_SECRET")
if missing_secops_auth:
    raise ValueError(
        "Missing Google SecOps auth values: " + ", ".join(missing_secops_auth)
    )

if webhook_has_key and webhook_has_secret:
    secops_auth_mode = "url_query"
elif (not webhook_has_key) and (not webhook_has_secret):
    secops_auth_mode = "headers"
else:
    secops_auth_mode = "mixed"

account_labels = [a["label"] for a in CONFIG["SNOWFLAKE_ACCOUNTS"]]
print("Config loaded.")
print(f"  Project dir : {PROJECT_DIR}")
print(
    f"  Env path    : {ENV_PATH} "
    f"({'found' if ENV_PATH.exists() else 'not found; relying on environment variables / prompts'})"
)
print(f"  Accounts    : {account_labels}")
print(f"  Lookback    : {CONFIG['LOOKBACK_DAYS']} days")
print(f"  Batch size  : {CONFIG['BATCH_SIZE']}")
print(f"  DRY_RUN     : {CONFIG['DRY_RUN']}")
print(f"  SecOps auth : {secops_auth_mode}")
print(f"  Export path : {CONFIG['EXPORT_PATH']}")


## Step 2 — Snowflake Connection

使用 `snowflake-connector-python` + key pair (`SNOWFLAKE_JWT`) 認證。

每個帳號連線後會確認 `SNOWFLAKE.TRUST_CENTER` schema 存在，若不存在則早點 fail 並提示原因。


In [ ]:
# Cell 6 - Snowflake connection factory + sanity check
from pathlib import Path

import snowflake.connector


def connect_snowflake(acct_cfg: dict) -> snowflake.connector.SnowflakeConnection:
    """Create a Snowflake connection and verify that the Trust Center schema exists."""
    private_key_path = Path(acct_cfg["private_key_path"]).expanduser().resolve()
    if not private_key_path.exists():
        raise FileNotFoundError(
            f"[{acct_cfg['label']}] Private key file not found: {private_key_path}"
        )

    conn = snowflake.connector.connect(
        account=acct_cfg["account"],
        user=acct_cfg["user"],
        authenticator="SNOWFLAKE_JWT",
        private_key_file=str(private_key_path),
        private_key_file_pwd=acct_cfg.get("private_key_passphrase") or None,
        role=acct_cfg["role"],
        warehouse=acct_cfg["warehouse"],
        client_session_keep_alive=True,
        network_timeout=30,
    )
    with conn.cursor() as cur:
        # Basic connection check.
        cur.execute("SELECT CURRENT_ACCOUNT(), CURRENT_ROLE(), CURRENT_VERSION()")
        row = cur.fetchone()
        print(f"  [{acct_cfg['label']}] account={row[0]}, role={row[1]}, version={row[2]}")

        # Trust Center schema check.
        cur.execute("SHOW SCHEMAS IN DATABASE SNOWFLAKE LIKE 'TRUST_CENTER'")
        if not cur.fetchall():
            raise RuntimeError(
                f"[{acct_cfg['label']}] SNOWFLAKE.TRUST_CENTER schema not found.\n"
                "  -> Check: (1) Snowflake Enterprise+ edition, (2) Trust Center enabled, "
                "(3) role has TRUST_CENTER_VIEWER grant"
            )
        print(f"  [{acct_cfg['label']}] Trust Center schema: OK")
    return conn


# Test the first account only. The fetch step opens and closes its own connections.
print("Testing connections...")
_test_conn = connect_snowflake(CONFIG["SNOWFLAKE_ACCOUNTS"][0])
try:
    print("Connection test passed.")
finally:
    _test_conn.close()


### (Optional) Snowflake REST SQL API v2

key pair 搭配 SQL API 需要自行簽 JWT。這個 notebook 預設走官方 Python connector，
因為 connector 已經幫你處理 `SNOWFLAKE_JWT` 的 login 細節。


In [ ]:
# Cell 8 - (REFERENCE ONLY - not executed) Snowflake REST SQL API v2 with key pair JWT
#
# The SQL API path is intentionally omitted here because it requires
# building and signing a JWT before every login request. The official
# Python connector already handles SNOWFLAKE_JWT authentication cleanly.

print("(REST API reference cell - skipped; use the Python connector for SNOWFLAKE_JWT auth)")


## Step 3 — Fetch Trust Center Findings (All Accounts)

對每個帳號建立連線，抓取 `SNOWFLAKE.TRUST_CENTER.FINDINGS`，加上 `ACCOUNT_NAME` 標籤後合併成統一 DataFrame。

若某帳號失敗，會印出 warning 並繼續處理其他帳號。

In [ ]:
# Cell 10 - Multi-account fetch
import atexit
import pandas as pd
import warnings

FINDINGS_COLUMNS = [
    "EVENT_ID",
    "OCCURRED_AT",
    "COMPLETED_AT",
    "SCANNER_ID",
    "SCANNER_NAME",
    "SCANNER_PACKAGE_ID",
    "SCANNER_SHORT_DESCRIPTION",
    "SCANNER_DESCRIPTION",
    "SEVERITY",
    "AT_RISK_ENTITIES",
    "RECOMMENDATION",
    "FINDING_DATA",
]
FINDINGS_OUTPUT_COLUMNS = FINDINGS_COLUMNS + ["ACCOUNT_NAME"]


def _empty_raw_findings_frame() -> pd.DataFrame:
    """Return an empty findings DataFrame without account metadata."""
    return pd.DataFrame(columns=FINDINGS_COLUMNS)


def _empty_findings_frame() -> pd.DataFrame:
    """Return a schema-stable findings DataFrame."""
    return pd.DataFrame(columns=FINDINGS_OUTPUT_COLUMNS)


if "close_all_connections" in globals():
    close_all_connections(quiet=True)

ACTIVE_CONNECTIONS = {}


def _close_connection(label: str, conn, quiet: bool = False) -> None:
    """Close one Snowflake connection and remove it from the active registry."""
    ACTIVE_CONNECTIONS.pop(label, None)
    try:
        conn.close()
        if not quiet:
            print(f"[{label}] connection closed")
    except Exception as e:
        if not quiet:
            print(f"[{label}] close error: {e}")


def close_all_connections(quiet: bool = False) -> None:
    """Close all currently registered Snowflake connections."""
    for label, conn in list(ACTIVE_CONNECTIONS.items()):
        _close_connection(label, conn, quiet=quiet)


if not globals().get("_connections_cleanup_registered", False):
    atexit.register(lambda: close_all_connections(quiet=True))
    _connections_cleanup_registered = True

TRUST_CENTER_QUERY = """
SELECT
    EVENT_ID,
    OCCURRED_AT,
    COMPLETED_AT,
    SCANNER_ID,
    SCANNER_NAME,
    SCANNER_PACKAGE_ID,
    SCANNER_SHORT_DESCRIPTION,
    SCANNER_DESCRIPTION,
    SEVERITY,
    AT_RISK_ENTITIES,
    RECOMMENDATION,
    FINDING_DATA
FROM SNOWFLAKE.TRUST_CENTER.FINDINGS
WHERE COALESCE(COMPLETED_AT, OCCURRED_AT) >= DATEADD(day, -{lookback}, CURRENT_TIMESTAMP())
ORDER BY COALESCE(COMPLETED_AT, OCCURRED_AT) DESC
"""


def _fetch_from_conn(
    conn: snowflake.connector.SnowflakeConnection, lookback_days: int
) -> pd.DataFrame:
    query = TRUST_CENTER_QUERY.format(lookback=int(lookback_days))
    with conn.cursor() as cur:
        cur.execute(query)
        frames = []
        try:
            for batch in cur.fetch_pandas_batches():
                frames.append(batch.reindex(columns=FINDINGS_COLUMNS))
        except Exception:
            if not frames:
                rows = cur.fetchall()
                cols = [c[0] for c in cur.description]
                frames = [pd.DataFrame(rows, columns=cols).reindex(columns=FINDINGS_COLUMNS)]

    if not frames:
        return _empty_raw_findings_frame()
    return pd.concat(frames, ignore_index=True).reindex(columns=FINDINGS_COLUMNS)


def fetch_findings_all_accounts(config: dict) -> tuple[pd.DataFrame, dict]:
    frames = []
    fetch_report = {
        "successful_accounts": [],
        "empty_accounts": [],
        "failed_accounts": {},
    }

    for acct in config["SNOWFLAKE_ACCOUNTS"]:
        conn = None
        label = acct["label"]
        try:
            print(f"Connecting to [{label}] {acct['account']}...")
            conn = connect_snowflake(acct)
            ACTIVE_CONNECTIONS[label] = conn
            df = _fetch_from_conn(conn, config["LOOKBACK_DAYS"]).copy()
            df["ACCOUNT_NAME"] = label
            frames.append(df.reindex(columns=FINDINGS_OUTPUT_COLUMNS))
            fetch_report["successful_accounts"].append(label)
            if df.empty:
                fetch_report["empty_accounts"].append(label)
            print(f"  -> {len(df)} findings")
        except Exception as e:
            fetch_report["failed_accounts"][label] = str(e)
            warnings.warn(f"[{label}] fetch failed: {e}")
        finally:
            if conn is not None:
                _close_connection(label, conn, quiet=True)

    if not fetch_report["successful_accounts"]:
        raise RuntimeError("All accounts failed. Check credentials and Trust Center permissions.")

    if frames:
        all_df = pd.concat(frames, ignore_index=True).reindex(columns=FINDINGS_OUTPUT_COLUMNS)
    else:
        all_df = _empty_findings_frame()

    print(
        f"\nTotal: {len(all_df)} findings across "
        f"{len(fetch_report['successful_accounts'])} successful account(s)"
    )
    if fetch_report["empty_accounts"]:
        print(f"Accounts with 0 findings: {fetch_report['empty_accounts']}")
    if fetch_report["failed_accounts"]:
        print("Accounts with fetch failures:")
        for failed_label, error in fetch_report["failed_accounts"].items():
            print(f"  - {failed_label}: {error}")

    return all_df, fetch_report


findings_df, fetch_report = fetch_findings_all_accounts(CONFIG)
if findings_df.empty:
    print("\nNo findings returned for the requested lookback window.")
else:
    print("\nSample:")
    print(findings_df[["ACCOUNT_NAME", "SCANNER_NAME", "SEVERITY", "COMPLETED_AT"]].head(10))


## Step 4 — Parse AT_RISK_ENTITIES

Trust Center 的 `AT_RISK_ENTITIES` 是 JSON array，每個 element 代表一個有風險的 entity（user、role、stored procedure 等）。

這裡把它展開成每 entity 一列，方便分析與匯出。

In [ ]:
# Cell 12 - AT_RISK_ENTITIES parser
import json

ENTITY_COLUMNS = [
    "ACCOUNT_NAME",
    "EVENT_ID",
    "SCANNER_NAME",
    "SCANNER_SHORT_DESCRIPTION",
    "SEVERITY",
    "ENTITY_TYPE",
    "ENTITY_NAME",
    "ENTITY_ID",
    "RAW_JSON",
]


def _safe_json(v):
    """Safely parse JSON values while tolerating dict/list/str/None inputs."""
    if v is None:
        return None
    if isinstance(v, (dict, list)):
        return v
    try:
        if pd.isna(v):
            return None
    except TypeError:
        pass
    try:
        return json.loads(v)
    except Exception:
        return {"raw": str(v)}


def parse_at_risk_entities(df: pd.DataFrame) -> pd.DataFrame:
    """
    Expand AT_RISK_ENTITIES JSON arrays into one row per entity.

    Output columns:
      ACCOUNT_NAME, EVENT_ID, SCANNER_NAME, SEVERITY,
      ENTITY_TYPE, ENTITY_NAME, ENTITY_ID, RAW_JSON
    """
    if df.empty:
        return pd.DataFrame(columns=ENTITY_COLUMNS)

    rows = []
    for _, r in df.iterrows():
        base = {
            "ACCOUNT_NAME": r.get("ACCOUNT_NAME"),
            "EVENT_ID": r.get("EVENT_ID"),
            "SCANNER_NAME": r.get("SCANNER_NAME"),
            "SCANNER_SHORT_DESCRIPTION": r.get("SCANNER_SHORT_DESCRIPTION"),
            "SEVERITY": r.get("SEVERITY"),
        }
        raw = r.get("AT_RISK_ENTITIES")
        entities = _safe_json(raw)
        if entities is None:
            entities = []
        elif isinstance(entities, dict):
            # Some scanners return a single object instead of an array.
            entities = [entities]
        elif not isinstance(entities, list):
            entities = [{"raw": str(entities)}]

        if not entities:
            rows.append(
                {
                    **base,
                    "ENTITY_TYPE": None,
                    "ENTITY_NAME": None,
                    "ENTITY_ID": None,
                    "RAW_JSON": None,
                }
            )
            continue

        for e in entities:
            if isinstance(e, str):
                e = {"name": e}
            elif not isinstance(e, dict):
                e = {"raw": str(e)}

            # Try several common key names because scanner payloads are not consistent.
            etype = (
                e.get("type")
                or e.get("entity_type")
                or e.get("objectType")
                or e.get("object_type")
                or ""
            ).upper() or None

            ename = (
                e.get("name")
                or e.get("display_name")
                or e.get("login_name")
                or e.get("userName")
                or e.get("user_name")
                or e.get("roleName")
                or e.get("role_name")
                or e.get("objectName")
                or e.get("object_name")
            )

            eid = (
                e.get("id")
                or e.get("user_id")
                or e.get("userId")
                or e.get("role_id")
                or e.get("roleId")
                or e.get("object_id")
                or e.get("objectId")
            )

            rows.append(
                {
                    **base,
                    "ENTITY_TYPE": etype,
                    "ENTITY_NAME": str(ename) if ename is not None else None,
                    "ENTITY_ID": str(eid) if eid is not None else None,
                    "RAW_JSON": json.dumps(e, default=str),
                }
            )

    return pd.DataFrame(rows, columns=ENTITY_COLUMNS)


entities_df = parse_at_risk_entities(findings_df)
print(f"AT_RISK_ENTITIES: {len(entities_df)} rows expanded from {len(findings_df)} findings")
if entities_df.empty:
    print("No entity details were expanded.")
else:
    print("\nEntity type breakdown:")
    print(entities_df["ENTITY_TYPE"].value_counts(dropna=False).to_string())
    print("\nSample:")
    print(
        entities_df[
            ["ACCOUNT_NAME", "SCANNER_NAME", "SEVERITY", "ENTITY_TYPE", "ENTITY_NAME"]
        ].head(10)
    )


## Step 5 — Transform to Google SecOps UDM

將 Snowflake Trust Center findings 轉換成 Chronicle UDM (Unified Data Model) 格式。

每筆 finding 對應一個 top-level UDM event，`AT_RISK_ENTITIES` 對應 `about[]` 陣列。
Sender 會把這些 event 序列化成 newline-delimited JSON，直接送到 `:importPushLogs` webhook feed。


In [ ]:
# Cell 14 - UDM transformer
import json

from datetime import datetime, timezone

SEVERITY_MAP = {
    "CRITICAL": "CRITICAL",
    "HIGH": "HIGH",
    "MEDIUM": "MEDIUM",
    "LOW": "LOW",
    "INFO": "INFORMATIONAL",
    "INFORMATIONAL": "INFORMATIONAL",
}


def _to_iso(ts) -> str | None:
    """Normalize timestamps to ISO 8601 or return None when unavailable."""
    if ts is None:
        return None
    try:
        if pd.isna(ts):
            return None
    except TypeError:
        pass
    parsed = pd.to_datetime(ts, utc=True, errors="coerce")
    if pd.isna(parsed):
        return None
    return parsed.isoformat()


def _entities_to_about(raw) -> list:
    entities = _safe_json(raw) or []
    if isinstance(entities, dict):
        entities = [entities]

    about = []
    for e in entities if isinstance(entities, list) else []:
        if isinstance(e, str):
            e = {"name": e}
        if not isinstance(e, dict):
            continue

        etype = (
            e.get("type")
            or e.get("entity_type")
            or e.get("objectType")
            or e.get("object_type")
            or ""
        ).upper()
        name = (
            e.get("name")
            or e.get("display_name")
            or e.get("login_name")
            or e.get("roleName")
            or e.get("objectName")
            or "unknown"
        )

        entry = {"resource": {"name": str(name), "resource_subtype": etype}}
        if etype in ("USER", "SERVICE_USER"):
            entry["user"] = {"userid": str(name)}
        elif etype == "ROLE":
            entry["group"] = {"group_display_name": str(name)}
        about.append(entry)

    return about


def finding_to_udm(row: dict, account_label: str, event_timestamp: str) -> dict:
    event_id = str(
        row.get("EVENT_ID")
        or f"{row.get('SCANNER_ID')}-{row.get('COMPLETED_AT') or row.get('OCCURRED_AT')}"
    )
    sev = SEVERITY_MAP.get(str(row.get("SEVERITY", "")).upper(), "UNKNOWN_SEVERITY")
    acct = str(row.get("ACCOUNT_NAME") or account_label)

    return {
        "metadata": {
            "event_timestamp": event_timestamp,
            "collected_timestamp": datetime.now(timezone.utc).isoformat(),
            "event_type": "SCAN_UNCATEGORIZED",
            "vendor_name": "Snowflake",
            "product_name": "Snowflake Trust Center",
            "product_log_id": event_id,
            "description": str(row.get("SCANNER_SHORT_DESCRIPTION") or ""),
            "ingestion_labels": [
                {"key": "snowflake_account", "value": acct},
                {
                    "key": "scanner_package",
                    "value": str(row.get("SCANNER_PACKAGE_ID") or ""),
                },
                {"key": "namespace", "value": "snowflake_trust_center"},
            ],
        },
        "principal": {
            "hostname": f"{acct}.snowflakecomputing.com",
        },
        "security_result": [
            {
                "category": "POLICY_VIOLATION",
                "rule_id": str(row.get("SCANNER_ID") or ""),
                "rule_name": str(row.get("SCANNER_NAME") or ""),
                "summary": str(row.get("SCANNER_SHORT_DESCRIPTION") or ""),
                "description": str(row.get("SCANNER_DESCRIPTION") or ""),
                "severity": sev,
                "action_details": str(row.get("RECOMMENDATION") or ""),
            }
        ],
        "about": _entities_to_about(row.get("AT_RISK_ENTITIES")),
        "additional": {"finding_data": _safe_json(row.get("FINDING_DATA")) or {}},
    }


def build_udm_events(findings: pd.DataFrame, default_account_label: str) -> tuple[list, dict]:
    """Build webhook-ready UDM events and record validation skips."""
    events = []
    validation = {
        "built": 0,
        "skipped_missing_event_timestamp": 0,
        "skipped_examples": [],
    }

    for _, record in findings.iterrows():
        row = record.to_dict()
        event_timestamp = _to_iso(row.get("COMPLETED_AT")) or _to_iso(row.get("OCCURRED_AT"))
        if not event_timestamp:
            validation["skipped_missing_event_timestamp"] += 1
            if len(validation["skipped_examples"]) < 5:
                validation["skipped_examples"].append(
                    {
                        "ACCOUNT_NAME": row.get("ACCOUNT_NAME"),
                        "EVENT_ID": row.get("EVENT_ID"),
                        "SCANNER_NAME": row.get("SCANNER_NAME"),
                    }
                )
            continue
        events.append(finding_to_udm(row, default_account_label, event_timestamp))

    validation["built"] = len(events)
    return events, validation


udm_events, udm_validation = build_udm_events(
    findings_df, CONFIG["SNOWFLAKE_ACCOUNTS"][0]["label"]
)
print(f"Transformed {len(udm_events)} webhook-ready UDM events")
if udm_validation["skipped_missing_event_timestamp"]:
    print(
        f"Skipped {udm_validation['skipped_missing_event_timestamp']} finding(s) "
        "because COMPLETED_AT and OCCURRED_AT were both missing."
    )
    print(json.dumps(udm_validation["skipped_examples"], indent=2, default=str))
if udm_events:
    print("\nFirst event sample:")
    print(json.dumps(udm_events[0], indent=2, default=str))


## Step 6 — Charts

4 張 matplotlib 圖表：
1. **Severity Distribution** — 各嚴重等級 finding 數（多帳號分組）
2. **Top 10 Scanners** — 觸發最多 findings 的 scanner（多帳號堆疊）
3. **Account × Severity Heatmap** — 帳號與嚴重等級交叉分析
4. **Finding Trend** — 每天 finding 數趨勢（多帳號折線）

In [ ]:
# Cell 16 - Charts
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")

SEVERITY_ORDER = ["CRITICAL", "HIGH", "MEDIUM", "LOW", "INFO", "INFORMATIONAL"]
SEVERITY_COLORS = {
    "CRITICAL": "#d32f2f",
    "HIGH": "#f57c00",
    "MEDIUM": "#fbc02d",
    "LOW": "#388e3c",
    "INFO": "#0288d1",
    "INFORMATIONAL": "#0288d1",
}
ACCOUNT_PALETTE = plt.cm.tab10.colors


def _draw_empty_chart(ax, title: str, message: str) -> None:
    """Render a consistent placeholder chart for empty datasets."""
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.text(0.5, 0.5, message, ha="center", va="center", fontsize=12)
    ax.set_axis_off()


fig_objects = {}
accounts = findings_df.get("ACCOUNT_NAME", pd.Series(dtype="object")).dropna().unique().tolist()

if findings_df.empty:
    print("Skipped chart generation because no findings were returned.")
else:
    # 1. Severity distribution.
    sev_pivot = (
        findings_df.groupby(["ACCOUNT_NAME", "SEVERITY"])
        .size()
        .reset_index(name="count")
        .pivot(index="SEVERITY", columns="ACCOUNT_NAME", values="count")
        .reindex([s for s in SEVERITY_ORDER if s in findings_df["SEVERITY"].dropna().unique()])
        .fillna(0)
    )
    fig1, ax1 = plt.subplots(figsize=(10, 5))
    if sev_pivot.empty:
        _draw_empty_chart(ax1, "Findings by Severity", "No severity values were available.")
    else:
        x = np.arange(len(sev_pivot))
        bar_w = 0.8 / max(len(accounts), 1)
        for i, col in enumerate(sev_pivot.columns):
            offset = (i - len(accounts) / 2 + 0.5) * bar_w
            bars = ax1.bar(
                x + offset,
                sev_pivot[col],
                bar_w * 0.9,
                label=col,
                color=ACCOUNT_PALETTE[i % 10],
            )
            for b in bars:
                h = b.get_height()
                if h > 0:
                    ax1.text(
                        b.get_x() + b.get_width() / 2,
                        h + 0.1,
                        int(h),
                        ha="center",
                        va="bottom",
                        fontsize=8,
                    )
        ax1.set_xticks(x)
        ax1.set_xticklabels(sev_pivot.index, fontsize=11)
        ax1.set_title("Findings by Severity", fontsize=14, fontweight="bold")
        ax1.set_ylabel("Count")
        ax1.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        if len(accounts) > 1:
            ax1.legend(title="Account", loc="upper right")
    plt.tight_layout()
    fig_objects["severity"] = fig1
    plt.show()

    # 2. Top 10 scanners.
    top_scanners = findings_df.groupby(["SCANNER_NAME", "ACCOUNT_NAME"]).size().reset_index(name="count")
    top10_names = top_scanners.groupby("SCANNER_NAME")["count"].sum().nlargest(10).index.tolist()
    scanner_pivot = (
        top_scanners[top_scanners["SCANNER_NAME"].isin(top10_names)]
        .pivot(index="SCANNER_NAME", columns="ACCOUNT_NAME", values="count")
        .fillna(0)
    )
    fig2, ax2 = plt.subplots(figsize=(12, 5))
    if scanner_pivot.empty:
        _draw_empty_chart(ax2, "Top 10 Scanners by Finding Count", "No scanner names were available.")
    else:
        bottom = np.zeros(len(scanner_pivot))
        for i, col in enumerate(scanner_pivot.columns):
            vals = scanner_pivot[col].values
            ax2.barh(
                scanner_pivot.index,
                vals,
                left=bottom,
                label=col,
                color=ACCOUNT_PALETTE[i % 10],
            )
            bottom += vals
        ax2.set_title("Top 10 Scanners by Finding Count", fontsize=14, fontweight="bold")
        ax2.set_xlabel("Count")
        ax2.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax2.invert_yaxis()
        if len(accounts) > 1:
            ax2.legend(title="Account", loc="lower right")
    plt.tight_layout()
    fig_objects["scanners"] = fig2
    plt.show()

    # 3. Account x severity heatmap.
    heat_data = (
        findings_df.groupby(["ACCOUNT_NAME", "SEVERITY"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=[s for s in SEVERITY_ORDER if s in findings_df["SEVERITY"].dropna().unique()])
        .fillna(0)
    )
    fig3, ax3 = plt.subplots(figsize=(10, max(3, len(accounts) * 1.2)))
    if heat_data.empty:
        _draw_empty_chart(ax3, "Account x Severity Heatmap", "No severity heatmap data was available.")
    else:
        im = ax3.imshow(heat_data.values, cmap="YlOrRd", aspect="auto")
        ax3.set_xticks(range(len(heat_data.columns)))
        ax3.set_xticklabels(heat_data.columns, fontsize=11)
        ax3.set_yticks(range(len(heat_data.index)))
        ax3.set_yticklabels(heat_data.index, fontsize=10)
        for r in range(len(heat_data.index)):
            for c in range(len(heat_data.columns)):
                v = int(heat_data.values[r, c])
                ax3.text(
                    c,
                    r,
                    str(v),
                    ha="center",
                    va="center",
                    color="white" if v > heat_data.values.max() * 0.6 else "black",
                    fontsize=12,
                )
        plt.colorbar(im, ax=ax3, shrink=0.8)
        ax3.set_title("Account x Severity Heatmap", fontsize=14, fontweight="bold")
    plt.tight_layout()
    fig_objects["heatmap"] = fig3
    plt.show()

    # 4. Finding trend.
    trend_df = findings_df.copy()
    trend_df["DATE"] = pd.to_datetime(trend_df["COMPLETED_AT"], errors="coerce").dt.normalize()
    trend_pivot = (
        trend_df.dropna(subset=["DATE"])
        .groupby(["DATE", "ACCOUNT_NAME"])
        .size()
        .reset_index(name="count")
        .pivot(index="DATE", columns="ACCOUNT_NAME", values="count")
        .fillna(0)
        .sort_index()
    )
    fig4, ax4 = plt.subplots(figsize=(12, 4))
    if trend_pivot.empty:
        _draw_empty_chart(
            ax4,
            "Finding Trend (by Day)",
            "No valid COMPLETED_AT timestamps were available.",
        )
    else:
        for i, col in enumerate(trend_pivot.columns):
            ax4.plot(
                trend_pivot.index,
                trend_pivot[col],
                marker="o",
                label=col,
                color=ACCOUNT_PALETTE[i % 10],
            )
        ax4.set_title("Finding Trend (by Day)", fontsize=14, fontweight="bold")
        ax4.set_ylabel("Count")
        ax4.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        plt.xticks(rotation=30)
        if len(accounts) > 1:
            ax4.legend(title="Account", loc="upper right")
    plt.tight_layout()
    fig_objects["trend"] = fig4
    plt.show()

print(f"Generated {len(fig_objects)} chart figure(s).")


## Step 7 — Export Excel Report

輸出多 sheet 格式化報表，含嵌入圖表：

| Sheet | 內容 |
|-------|------|
| `README` | 報表說明、產生時間、帳號清單 |
| `All_Findings` | 所有帳號 findings 彙整 |
| `At_Risk_Entities` | AT_RISK_ENTITIES 展開 detail |
| `Severity_Summary` | pivot + 4 張圖表嵌入 |
| `Scanner_Summary` | scanner × 帳號 pivot |
| `{account_label}` | 每帳號獨立 sheet |

In [ ]:
# Cell 18 - Excel export
import io
import re
from datetime import datetime, timezone
from pathlib import Path

from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage
from openpyxl.styles import Alignment, Font
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# OpenPyXL helpers.


def _add_excel_table(ws, table_name: str):
    if ws.max_row <= 1 or ws.max_column == 0:
        return
    ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
    table = Table(displayName=table_name, ref=ref)
    table.tableStyleInfo = TableStyleInfo(
        name="TableStyleMedium2",
        showRowStripes=True,
        showFirstColumn=False,
        showLastColumn=False,
        showColumnStripes=False,
    )
    ws.add_table(table)


def _auto_col_widths(ws, max_width: int = 60):
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len = max((len(str(c.value or "")) for c in col), default=10)
        ws.column_dimensions[col_letter].width = min(max(12, max_len + 2), max_width)


def _wrap_column(ws, header_name: str):
    """Apply wrap_text to the requested column when it exists."""
    header_row = [c.value for c in ws[1]]
    if header_name not in header_row:
        return
    col_idx = header_row.index(header_name) + 1
    for row_idx in range(2, ws.max_row + 1):
        cell = ws.cell(row=row_idx, column=col_idx)
        cell.alignment = Alignment(wrap_text=True, vertical="top")


def _embed_chart(ws, fig, anchor_cell: str, scale: float = 0.85):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    buf.seek(0)
    img = XLImage(buf)
    img.width = int(img.width * scale)
    img.height = int(img.height * scale)
    ws.add_image(img, anchor_cell)


def _write_readme(ws, config: dict, findings_df: pd.DataFrame):
    ws.column_dimensions["A"].width = 20
    ws.column_dimensions["B"].width = 60
    rows = [
        ("Report", "Snowflake Trust Center to Google SecOps"),
        ("Generated At", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")),
        ("Lookback Days", str(config["LOOKBACK_DAYS"])),
        ("Accounts", ", ".join(a["label"] for a in config["SNOWFLAKE_ACCOUNTS"])),
        ("Total Findings", str(len(findings_df))),
        ("DRY_RUN", str(config["DRY_RUN"])),
        ("", ""),
        ("Sheets", "All_Findings - all accounts consolidated"),
        ("", "At_Risk_Entities - expanded entity details"),
        ("", "Severity_Summary - pivot and charts"),
        ("", "Scanner_Summary - scanner by account pivot"),
        ("", "<account_label> - per-account findings"),
    ]
    for r, (k, v) in enumerate(rows, start=1):
        ws.cell(r, 1, k).font = Font(bold=bool(k))
        ws.cell(r, 2, v)


# Main export function.


def export_to_excel(
    findings_df: pd.DataFrame,
    entities_df: pd.DataFrame,
    fig_objects: dict,
    config: dict,
    output_path: str,
):
    output_file = Path(output_path).expanduser().resolve()
    output_file.parent.mkdir(parents=True, exist_ok=True)

    export_cols = [
        "ACCOUNT_NAME",
        "EVENT_ID",
        "COMPLETED_AT",
        "OCCURRED_AT",
        "SCANNER_ID",
        "SCANNER_NAME",
        "SCANNER_PACKAGE_ID",
        "SCANNER_SHORT_DESCRIPTION",
        "SCANNER_DESCRIPTION",
        "SEVERITY",
        "RECOMMENDATION",
    ]
    export_df = findings_df[[c for c in export_cols if c in findings_df.columns]].copy()

    sev_summary = (
        findings_df.groupby(["ACCOUNT_NAME", "SEVERITY"])
        .size()
        .unstack(fill_value=0)
        .reindex(
            columns=[
                s
                for s in SEVERITY_ORDER
                if s in findings_df.get("SEVERITY", pd.Series(dtype="object")).dropna().unique()
            ]
        )
        .reset_index()
    )
    scanner_summary = (
        findings_df.groupby(["SCANNER_NAME", "ACCOUNT_NAME"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        pd.DataFrame({"Note": ["See README sheet"]}).to_excel(
            writer, sheet_name="README", index=False
        )

        export_df.to_excel(writer, sheet_name="All_Findings", index=False)

        entities_export = entities_df[
            [
                c
                for c in [
                    "ACCOUNT_NAME",
                    "EVENT_ID",
                    "SCANNER_NAME",
                    "SCANNER_SHORT_DESCRIPTION",
                    "SEVERITY",
                    "ENTITY_TYPE",
                    "ENTITY_NAME",
                    "ENTITY_ID",
                    "RAW_JSON",
                ]
                if c in entities_df.columns
            ]
        ]
        entities_export.to_excel(writer, sheet_name="At_Risk_Entities", index=False)

        sev_summary.to_excel(writer, sheet_name="Severity_Summary", index=False)
        scanner_summary.to_excel(writer, sheet_name="Scanner_Summary", index=False)

        for acct in config["SNOWFLAKE_ACCOUNTS"]:
            label = acct["label"]
            sub_df = export_df[export_df["ACCOUNT_NAME"] == label].copy()
            if not sub_df.empty:
                safe_label = re.sub(r'[\\/*?:"<>|\[\]]', "", label)[:31]
                sub_df.to_excel(writer, sheet_name=safe_label, index=False)

    wb = load_workbook(output_file)

    readme_ws = wb["README"]
    for row in readme_ws.iter_rows():
        for cell in row:
            cell.value = None
    _write_readme(readme_ws, config, findings_df)

    for sheet_name in [
        "All_Findings",
        "At_Risk_Entities",
        "Severity_Summary",
        "Scanner_Summary",
    ]:
        if sheet_name in wb.sheetnames:
            ws = wb[sheet_name]
            _auto_col_widths(ws)
            safe_name = sheet_name.replace(" ", "_").replace("-", "_")
            _add_excel_table(ws, f"Table_{safe_name}")

    for col_name in ["SCANNER_DESCRIPTION", "RECOMMENDATION", "RAW_JSON"]:
        for sheet_name in wb.sheetnames:
            _wrap_column(wb[sheet_name], col_name)

    for i, acct in enumerate(config["SNOWFLAKE_ACCOUNTS"]):
        safe_label = re.sub(r'[\\/*?:"<>|\[\]]', "", acct["label"])[:31]
        if safe_label in wb.sheetnames:
            ws = wb[safe_label]
            _auto_col_widths(ws)
            _add_excel_table(ws, f"Table_acct_{i}")

    sev_ws = wb["Severity_Summary"]
    anchor_row = sev_ws.max_row + 3
    chart_anchors = {
        "severity": f"A{anchor_row}",
        "heatmap": f"L{anchor_row}",
        "scanners": f"A{anchor_row + 22}",
        "trend": f"L{anchor_row + 22}",
    }
    for key, anchor in chart_anchors.items():
        if key in fig_objects:
            _embed_chart(sev_ws, fig_objects[key], anchor)

    wb.save(output_file)
    print(f"Excel report saved: {output_file}")
    return str(output_file)


excel_path = export_to_excel(
    findings_df=findings_df,
    entities_df=entities_df,
    fig_objects=fig_objects,
    config=CONFIG,
    output_path=CONFIG["EXPORT_PATH"],
)
print(f"Done: {excel_path}")


## Step 8 — Send to Google SecOps

透過 webhook POST UDM events 到 Google SecOps (`:importPushLogs`)。

- 每批最多 `BATCH_SIZE` 筆，且單次 request 會限制在 4 MB 內
- body 格式為 newline-delimited JSON，每行一筆 top-level UDM event
- 5xx / 408 / 429 自動 exponential backoff retry（4 次）
- 4xx 立即 raise（不 retry，避免掩蓋 auth 問題）
- `DRY_RUN=true` 只印批次規劃與 payload preview，不實際送出


In [ ]:
# Cell 20 - Webhook sender with batching + retry
import json
import logging
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from urllib.parse import parse_qs, urlparse

import requests
from tenacity import (
    before_sleep_log,
    retry,
    retry_if_exception_type,
    stop_after_attempt,
    wait_exponential,
)

logging.basicConfig(level=logging.WARNING)
_log = logging.getLogger("secops_sender")
_DEFAULT_SECOPS_WAIT = wait_exponential(multiplier=1, min=2, max=30)
_MAX_REQUEST_BYTES = (4 * 1024 * 1024) - 1024


class RetryableSecOpsError(Exception):
    def __init__(self, message: str, retry_after: int | None = None):
        super().__init__(message)
        self.retry_after = retry_after


class NonRetryableSecOpsError(Exception):
    pass


def _parse_retry_after(value: str | None) -> int | None:
    if not value:
        return None
    try:
        return max(0, int(value))
    except ValueError:
        pass
    try:
        retry_at = parsedate_to_datetime(value)
        if retry_at.tzinfo is None:
            retry_at = retry_at.replace(tzinfo=timezone.utc)
        return max(0, int((retry_at - datetime.now(timezone.utc)).total_seconds()))
    except Exception:
        return None


def _secops_wait(retry_state) -> float:
    exc = retry_state.outcome.exception() if retry_state.outcome else None
    if isinstance(exc, RetryableSecOpsError) and exc.retry_after is not None:
        return exc.retry_after
    return _DEFAULT_SECOPS_WAIT(retry_state)


def _url_has_query_credential(url: str, key: str) -> bool:
    values = parse_qs(urlparse(url).query).get(key, [])
    return any(value.strip() for value in values)


def _describe_auth_mode(url: str) -> str:
    has_key = _url_has_query_credential(url, "key")
    has_secret = _url_has_query_credential(url, "secret")
    if has_key and has_secret:
        return "url_query"
    if (not has_key) and (not has_secret):
        return "headers"
    return "mixed"


def _build_headers(config: dict) -> dict:
    headers = {
        "Content-Type": "application/json",
        "User-Agent": "snow-trust-center-secops/1.0",
    }
    if not _url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "key"):
        if not config.get("SECOPS_API_KEY"):
            raise NonRetryableSecOpsError(
                "SECOPS_API_KEY is required because the webhook URL does not include key=."
            )
        headers["X-goog-api-key"] = config["SECOPS_API_KEY"]
    if not _url_has_query_credential(config["SECOPS_WEBHOOK_URL"], "secret"):
        if not config.get("SECOPS_WEBHOOK_SECRET"):
            raise NonRetryableSecOpsError(
                "SECOPS_WEBHOOK_SECRET is required because the webhook URL does not include secret=."
            )
        headers["X-Webhook-Access-Key"] = config["SECOPS_WEBHOOK_SECRET"]
    return headers


def _serialize_event(event: dict) -> bytes:
    return json.dumps(event, separators=(",", ":"), ensure_ascii=False, default=str).encode(
        "utf-8"
    )


def _build_event_batches(events: list, batch_size: int) -> list[dict]:
    """Split events by count and request size, keeping each request under the feed limit."""
    if batch_size < 1:
        raise ValueError("BATCH_SIZE must be >= 1.")

    batches = []
    current_lines = []
    current_bytes = 0

    for event_index, event in enumerate(events, start=1):
        line = _serialize_event(event)
        if len(line) > _MAX_REQUEST_BYTES:
            raise NonRetryableSecOpsError(
                f"Event {event_index} is {len(line)} bytes, exceeding the 4 MB webhook limit."
            )

        line_bytes = len(line) + (1 if current_lines else 0)
        if current_lines and (
            len(current_lines) >= batch_size or current_bytes + line_bytes > _MAX_REQUEST_BYTES
        ):
            payload = b"\n".join(current_lines)
            batches.append(
                {
                    "events": len(current_lines),
                    "bytes": len(payload),
                    "payload": payload,
                }
            )
            current_lines = [line]
            current_bytes = len(line)
        else:
            current_lines.append(line)
            current_bytes += line_bytes

    if current_lines:
        payload = b"\n".join(current_lines)
        batches.append(
            {
                "events": len(current_lines),
                "bytes": len(payload),
                "payload": payload,
            }
        )

    return batches


@retry(
    stop=stop_after_attempt(4),
    wait=_secops_wait,
    retry=retry_if_exception_type((requests.RequestException, RetryableSecOpsError)),
    before_sleep=before_sleep_log(_log, logging.WARNING),
    reraise=True,
)
def _post_batch(url: str, headers: dict, payload: bytes) -> requests.Response:
    response = requests.post(url, headers=headers, data=payload, timeout=60)

    if response.status_code in (408, 429) or response.status_code >= 500:
        retry_after = _parse_retry_after(response.headers.get("Retry-After"))
        raise RetryableSecOpsError(
            f"Retryable error {response.status_code}: {response.text[:300]}",
            retry_after=retry_after,
        )
    if response.status_code >= 400:
        raise NonRetryableSecOpsError(
            f"Client error {response.status_code}: {response.text[:500]}\n"
            "  -> 401/403: check webhook URL query auth or fallback headers\n"
            "  -> 413: reduce BATCH_SIZE or trim oversized events"
        )
    return response


def send_to_secops(events: list, config: dict) -> dict:
    total = len(events)
    if total == 0:
        print("No UDM events to send; skipping SecOps delivery.")
        return {
            "sent": 0,
            "batches": 0,
            "attempted_batches": 0,
            "failed_batches": 0,
            "failed_event_count": 0,
            "planned_batches": 0,
            "planned_bytes": 0,
            "auth_mode": _describe_auth_mode(config["SECOPS_WEBHOOK_URL"]),
            "dry_run": config["DRY_RUN"],
            "skipped": True,
            "errors": [],
        }

    batches = _build_event_batches(events, config["BATCH_SIZE"])
    planned_bytes = sum(batch["bytes"] for batch in batches)
    auth_mode = _describe_auth_mode(config["SECOPS_WEBHOOK_URL"])

    if config["DRY_RUN"]:
        first_batch = batches[0]
        preview = first_batch["payload"][:1500].decode("utf-8", errors="replace")
        print(f"[DRY_RUN] {total} events would be sent to {config['SECOPS_WEBHOOK_URL']}")
        print(f"Auth mode   : {auth_mode}")
        print(f"Batches     : {len(batches)}")
        print(f"Total bytes : {planned_bytes}")
        print(
            f"First batch : {first_batch['events']} events / {first_batch['bytes']} bytes"
        )
        print("\nFirst batch payload preview:")
        print(preview)
        if len(first_batch["payload"]) > 1500:
            print("... <truncated>")
        return {
            "sent": 0,
            "batches": 0,
            "attempted_batches": 0,
            "failed_batches": 0,
            "failed_event_count": 0,
            "planned_batches": len(batches),
            "planned_bytes": planned_bytes,
            "auth_mode": auth_mode,
            "dry_run": True,
            "skipped": False,
            "errors": [],
        }

    headers = _build_headers(config)
    sent = 0
    successful_batches = 0
    failed_batches = 0
    failed_event_count = 0
    errors = []
    attempted_batches = 0

    for batch_index, batch in enumerate(batches, start=1):
        attempted_batches = batch_index
        try:
            _post_batch(config["SECOPS_WEBHOOK_URL"], headers, batch["payload"])
            sent += batch["events"]
            successful_batches += 1
            print(
                f"  Batch {batch_index}: sent {batch['events']} events / {batch['bytes']} bytes "
                f"(total {sent}/{total})"
            )
        except Exception as e:
            failed_batches += 1
            failed_event_count += batch["events"]
            errors.append(
                {
                    "batch": batch_index,
                    "events": batch["events"],
                    "bytes": batch["bytes"],
                    "error": str(e),
                }
            )
            print(f"  Batch {batch_index}: failed after retries - {e}")
            break

    if failed_batches:
        print(
            f"\nStopped after {sent} sent event(s); "
            f"{failed_event_count} event(s) remain unsent across {failed_batches} failed batch(es)."
        )
    else:
        print(f"\nComplete: {sent} events sent in {successful_batches} batch(es)")

    return {
        "sent": sent,
        "batches": successful_batches,
        "attempted_batches": attempted_batches,
        "failed_batches": failed_batches,
        "failed_event_count": failed_event_count,
        "planned_batches": len(batches),
        "planned_bytes": planned_bytes,
        "auth_mode": auth_mode,
        "dry_run": False,
        "skipped": False,
        "errors": errors,
    }


secops_result = send_to_secops(udm_events, CONFIG)
secops_result


## Step 9 — Verification

確認各步驟輸出都正確，並提示如何在 Chronicle UI 查詢。


In [ ]:
# Cell 22 - Summary & verification
import json

from collections import Counter

severity_series = findings_df.get("SEVERITY", pd.Series(dtype="object")).fillna("UNKNOWN")
scanner_series = findings_df.get("SCANNER_NAME", pd.Series(dtype="object")).fillna("UNKNOWN")
processed_findings = len(udm_events) + udm_validation["skipped_missing_event_timestamp"]

summary = {
    "accounts_queried": [a["label"] for a in CONFIG["SNOWFLAKE_ACCOUNTS"]],
    "fetch_report": fetch_report,
    "lookback_days": CONFIG["LOOKBACK_DAYS"],
    "total_findings_fetched": len(findings_df),
    "severity_breakdown": dict(Counter(severity_series)),
    "scanner_breakdown": dict(Counter(scanner_series).most_common(10)),
    "entities_expanded": len(entities_df),
    "udm_events_built": len(udm_events),
    "udm_validation": udm_validation,
    "excel_exported_to": excel_path,
    "secops_result": secops_result,
}
print(json.dumps(summary, indent=2, default=str))

print("\n=== Verification ===")
checks = [
    (
        bool(fetch_report["successful_accounts"]),
        "at least one account fetched successfully",
        "All account fetches failed.",
    ),
    (
        not fetch_report["failed_accounts"],
        "all account fetches succeeded",
        f"Partial fetch failure: {list(fetch_report['failed_accounts'])}",
    ),
    (
        processed_findings == summary["total_findings_fetched"],
        "all findings accounted for during UDM build",
        f"Processed {processed_findings} / expected {summary['total_findings_fetched']}",
    ),
    (
        CONFIG["DRY_RUN"]
        or len(udm_events) == 0
        or secops_result["sent"] == len(udm_events),
        "all UDM events delivered to SecOps",
        f"Sent {secops_result['sent']} / expected {len(udm_events)}",
    ),
]
all_pass = True
for ok, msg, fail_msg in checks:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {msg}" if ok else f"  [{status}] {fail_msg}")
    if not ok:
        all_pass = False

if findings_df.empty:
    print("  [PASS] 0 findings returned; downstream steps ran in no-op mode.")
if udm_validation["skipped_missing_event_timestamp"]:
    print(
        f"  [WARN] Skipped {udm_validation['skipped_missing_event_timestamp']} finding(s) "
        "because event timestamps were missing."
    )

if all_pass:
    print("\nAll checks passed.")

print("\n=== Chronicle UI Query ===")
print('metadata.product_name = "Snowflake Trust Center"')
query_accounts = fetch_report["successful_accounts"] or [
    a["label"] for a in CONFIG["SNOWFLAKE_ACCOUNTS"]
]
for account_label in query_accounts:
    print(f'metadata.ingestion_labels["snowflake_account"] = "{account_label}"')


## Cleanup

In [ ]:
# Cell 24 - Close all Snowflake connections
close_all_connections()
if not ACTIVE_CONNECTIONS:
    print("No open Snowflake connections remain.")


---

## Troubleshooting

### Snowflake

| 問題 | 解法 |
|------|------|
| `Private key file not found` | 確認 `SNOWFLAKE_PRIVATE_KEY_PATH_<n>` 指到正確 `.p8` 檔案，並檢查本機檔案權限 |
| `JWT token is invalid` / `JWT token authentication failed` | 確認 private key 對應的 public key 已綁到使用者，並檢查 user / account 是否正確 |
| `Could not deserialize key data` / `bad decrypt` | 確認 `SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_<n>` 是否正確，或 private key 是否為支援格式 |
| `TRUST_CENTER schema not found` | 確認 Snowflake Enterprise+ edition、Trust Center 已啟用、role 有 `TRUST_CENTER_VIEWER` grant |
| `Warehouse not running` | `ALTER WAREHOUSE <wh> RESUME;` |
| `No findings fetched` | 增加 `LOOKBACK_DAYS` 或在 Snowflake Trust Center UI 確認 scanners 有執行 |
| `AT_RISK_ENTITIES empty` | 此 scanner 的 finding 可能尚未有受影響 entity，屬正常情況 |

### Google SecOps Webhook

| 問題 | 解法 |
|------|------|
| `401 Unauthorized` | 確認 `SECOPS_WEBHOOK_URL` 內的 `key=` 正確，或 `SECOPS_API_KEY` fallback header 值正確 |
| `403 Forbidden` | 確認 `secret=` 或 `SECOPS_WEBHOOK_SECRET` 正確，並確認 API key / secret 配對的是同一個 feed |
| `413 Request Too Large` | 減少 `BATCH_SIZE`，或檢查單筆 event 是否過大 |
| Events 未出現在 Chronicle UI | 等 2-5 分鐘索引；確認搜尋時間範圍涵蓋 `COMPLETED_AT` |
| 送出格式不符 | 確認 body 是 newline-delimited JSON，且每行都是 top-level UDM event |

### Excel

| 問題 | 解法 |
|------|------|
| 檔案被佔用無法寫入 | 關閉已開啟的 Excel 視窗，或修改 `EXPORT_PATH` |
| 圖表未嵌入 | 確認 Cell 16 有先執行（`fig_objects` 不為空） |
| Table name 重複錯誤 | 清除舊的 xlsx 後重新執行 |

### 多帳號

| 問題 | 解法 |
|------|------|
| 部分帳號失敗不想影響其他帳號 | 已內建 try/except，失敗帳號會印 warning 並繼續 |
| 要加第 4 個帳號 | 在 `.env` 加 `SNOWFLAKE_ACCOUNT_4`、`SNOWFLAKE_PRIVATE_KEY_PATH_4` 等對應變數即可 |
